# 14.7 — Deep learning for time series

Deep time-series models learn forecast functions from windows, covariates, and many related histories.

Deep forecasting turns chronological windows into supervised rows, then learns a shared forecast rule. This CPU-only notebook uses NumPy linear window blocks to show the same data discipline without training a heavy neural net. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1306
rng = np.random.default_rng(SEED)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def interval_coverage(y_true, lower, upper):
    y_true = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    inside = (y_true >= lower) & (y_true <= upper)
    return float(np.mean(inside))


def f13_time_series_ladder():
    t = np.arange(72, dtype=float)
    rungs = []

    y1 = np.full_like(t, 10.0)
    rungs.append({"name": "D1 constant", "t": t, "y": y1, "true": y1, "period": 12})

    y2 = 8.0 + 0.18 * t
    rungs.append({"name": "D2 drift", "t": t, "y": y2, "true": y2, "period": 12})

    noise3 = rng.normal(0.0, 0.45, size=t.size)
    true3 = 8.0 + 0.18 * t
    y3 = true3 + noise3
    rungs.append({"name": "D3 drift + noise", "t": t, "y": y3, "true": true3, "period": 12})

    seasonal4 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true4 = 8.0 + 0.12 * t + seasonal4
    y4 = true4 + rng.normal(0.0, 0.35, size=t.size)
    rungs.append({"name": "D4 seasonal", "t": t, "y": y4, "true": true4, "period": 12})

    seasonal5 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true5 = 8.0 + 0.10 * t + seasonal5
    true5 = true5 + np.where(t >= 48.0, 3.0 + 0.06 * (t - 48.0), 0.0)
    y5 = true5 + rng.normal(0.0, 0.45, size=t.size)
    y5[18] = y5[18] + 5.0
    y5[55] = y5[55] - 4.0
    rungs.append({"name": "D5 outliers + regime shift", "t": t, "y": y5, "true": true5, "period": 12})

    return rungs


def train_test_split_series(rung, train_size=48):
    train = {"t": rung["t"][:train_size], "y": rung["y"][:train_size]}
    test = {"t": rung["t"][train_size:], "y": rung["y"][train_size:], "true": rung["true"][train_size:]}
    return train, test


def print_ladder_preview(rungs):
    for rung in rungs:
        name = rung["name"]
        y = rung["y"]
        sample = np.round(y[:6], 3)
        print(f"{name}: n={y.size}, period={rung['period']}, first six={sample}")


def plot_forecast_panels(results, metric_name="RMSE", marker_key=None, component_key=None):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.ravel()

    for idx, result in enumerate(results):
        ax = axes[idx]
        ax.plot(result["test_t"], result["test_y"], label="observed", color="black")
        ax.plot(result["test_t"], result["forecast"], label="forecast", color="tab:blue")
        ax.plot(result["test_t"], result["test_true"], label="true signal", color="tab:green", alpha=0.7)
        if component_key is not None and component_key in result:
            ax.plot(result["test_t"], result[component_key], label=component_key, color="tab:orange", alpha=0.8)
        if marker_key is not None and marker_key in result:
            marks = result[marker_key]
            for mark in np.atleast_1d(marks):
                ax.axvline(mark, color="tab:red", linestyle="--", alpha=0.6)
        ax.set_title(f"{result['name']}\n{metric_name}={result['metric']:.3f}")
        ax.grid(True, alpha=0.25)

    axes[-1].plot(range(1, len(results) + 1), [r["metric"] for r in results], marker="o")
    axes[-1].set_xticks(range(1, len(results) + 1))
    axes[-1].set_xlabel("D-rung")
    axes[-1].set_ylabel(metric_name)
    axes[-1].set_title(f"{metric_name} curve")
    axes[-1].grid(True, alpha=0.25)
    axes[0].legend(loc="best", fontsize=8)
    fig.tight_layout()



## The concept, built once (D1)

A deep forecaster learns a function from a window to a horizon:

$$\hat y_{t+1:t+H}=f_\theta(x_{t-L+1:t},c_{t+1:t+H})$$

Here we build the linear window block that sits underneath many neural examples. It must reproduce the lesson weighted prediction $12.600$, shared-slope forecasts $\{16,26,36\}$, and N-BEATS-style backcast MSE $0.666667$.


In [ ]:

def window_forecaster(window, weights, bias=0.0):
    window = np.asarray(window, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return float(window @ weights + bias)

prediction = window_forecaster([10.0, 12.0, 14.0], [0.2, 0.3, 0.5])
series = np.array([
    [10.0, 12.0, 14.0, 15.0],
    [20.0, 22.0, 24.0, 25.0],
    [30.0, 32.0, 34.0, 35.0],
])
increments = np.diff(series, axis=1)
shared_slope = float(np.mean(increments[:, -1]))
shared_forecast = series[:, -1] + shared_slope
past = np.array([10.0, 12.0, 11.0])
backcast = np.array([10.0, 11.0, 12.0])
backcast_mse = float(np.mean((past - backcast) ** 2))

assert np.isclose(prediction, 12.600, atol=1e-12)
assert np.allclose(shared_forecast, [16.0, 26.0, 36.0], atol=1e-12)
assert np.isclose(backcast_mse, 0.666667, atol=5e-7)
print("lesson checks passed")



To make the block usable on the ladder, convert history into chronological windows. A ridge fit estimates window weights; the code never trains a large neural network, but it preserves the same no-leakage supervised framing.


In [ ]:

def make_windows(y, lookback=12, start=0, stop=None):
    y = np.asarray(y, dtype=float)
    if stop is None:
        stop = y.size
    X = []
    target = []
    times = []
    for idx in range(max(lookback, start), stop):
        X.append(y[idx - lookback:idx])
        target.append(y[idx])
        times.append(idx)
    return np.asarray(X), np.asarray(target), np.asarray(times)


def fit_window_ridge(y, lookback=12, ridge=1.0):
    X, target, times = make_windows(y, lookback=lookback)
    design = np.column_stack([np.ones(X.shape[0]), X])
    penalty = np.eye(design.shape[1]) * ridge
    penalty[0, 0] = 0.0
    beta = np.linalg.solve(design.T @ design + penalty, design.T @ target)

    def predict_from_history(history, horizon):
        history = list(np.asarray(history, dtype=float))
        preds = []
        for step in range(horizon):
            window = np.asarray(history[-lookback:], dtype=float)
            pred = float(beta[0] + window @ beta[1:])
            preds.append(pred)
            history.append(pred)
        return np.asarray(preds)

    return {"beta": beta, "predict_from_history": predict_from_history, "times": times}



## The dataset ladder

We use the same F13 ladder in every notebook so the method faces increasing time-series difficulty inline: D1 constant, D2 drift, D3 drift plus noise, D4 monthly seasonality, and D5 outliers plus a real regime shift.


In [ ]:

rungs = f13_time_series_ladder()
print_ladder_preview(rungs)


In [ ]:

results = []

for rung in rungs:
    train, test = train_test_split_series(rung)
    model = fit_window_ridge(train["y"], lookback=12, ridge=2.0)
    forecast = model["predict_from_history"](train["y"], horizon=test["y"].size)
    residual_scale = float(np.std(train["y"][12:] - model["predict_from_history"](train["y"][:12], horizon=train["y"].size - 12)))
    lower = forecast - 1.2816 * residual_scale
    upper = forecast + 1.2816 * residual_scale
    metric = rmse(test["true"], forecast)
    cov = interval_coverage(test["y"], lower, upper)
    results.append({
        "name": rung["name"],
        "test_t": test["t"],
        "test_y": test["y"],
        "test_true": test["true"],
        "forecast": forecast,
        "metric": metric,
        "coverage": cov,
    })

for result in results:
    print(f"{result['name']}: RMSE={result['metric']:.3f}, 80% coverage={result['coverage']:.3f}")


In [ ]:

plot_forecast_panels(results, metric_name="RMSE")



## Pitfall on D5: windows that cross validation boundaries

If validation windows are created after concatenating future data, the model sees regime-shift information while pretending to forecast it. The fix is to train windows only on times before the origin.


In [ ]:

d5 = rungs[-1]
train, test = train_test_split_series(d5)
leaky_model = fit_window_ridge(d5["y"], lookback=12, ridge=2.0)
leaky_windows, leaky_target, leaky_times = make_windows(d5["y"], lookback=12, start=48, stop=72)
leaky_design = np.column_stack([np.ones(leaky_windows.shape[0]), leaky_windows])
leaky_forecast = leaky_design @ leaky_model["beta"]
honest_model = fit_window_ridge(train["y"], lookback=12, ridge=2.0)
honest_forecast = honest_model["predict_from_history"](train["y"], horizon=test["y"].size)
leaky_rmse = rmse(test["true"], leaky_forecast)
honest_rmse = rmse(test["true"], honest_forecast)
print(f"leaky validation RMSE={leaky_rmse:.3f}")
print(f"chronological RMSE={honest_rmse:.3f}")
print("The leaky score is not a valid forecast audit even when it is smaller.")



## Evaluate it + Practice

- Compare rolling-window RMSE against a last-value baseline; a learned window should help on drift and seasonality.
- Sanity check: windows must be created chronologically, never across the validation boundary.
- Ablation: train only local weights per rung; small noisy histories should become less stable than the pooled rule.
- Failure signal: validation RMSE is unrealistically tiny while chronological plots lag or jump at the split.
- For probabilistic heads, add interval coverage in addition to point RMSE.

Practice prompts:
1. Change the lookback from 6 to 12 and predict which rung benefits most.


2. Modify the hardest rung and rerun the same metric table.

3. Write one sentence explaining whether RMSE or coverage is the better decision metric here.